# 01 - Metrics Anomaly Detection

Notebook này phân tích metrics của `cart-service`, `api-gateway`, `order-service`, `payment-service`, `product-service` để trả lời câu hỏi WHEN.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

BASE = Path(r"D:\\AWS\\AIOPS-study\\g2-data\\g2\\metrics")
ART = Path(r"D:\\AWS\\AIOPS\\w1\\lab\\artifacts")
ART.mkdir(parents=True, exist_ok=True)
plt.style.use('seaborn-v0_8')

In [ ]:
cart = pd.read_csv(BASE / 'cart-service.csv', parse_dates=['timestamp'])
api = pd.read_csv(BASE / 'api-gateway.csv', parse_dates=['timestamp'])
order = pd.read_csv(BASE / 'order-service.csv', parse_dates=['timestamp'])
payment = pd.read_csv(BASE / 'payment-service.csv', parse_dates=['timestamp'])
product = pd.read_csv(BASE / 'product-service.csv', parse_dates=['timestamp'])
cart.head()

In [ ]:
base = cart.iloc[:720]
cart['mem_z'] = (cart['memory_usage_bytes'] - base['memory_usage_bytes'].mean()) / base['memory_usage_bytes'].std()
cart['gc_z'] = (cart['jvm_gc_pause_ms_avg'] - base['jvm_gc_pause_ms_avg'].mean()) / base['jvm_gc_pause_ms_avg'].std()
cart['mem_sustained'] = cart['mem_z'].rolling(6).mean()
cart['gc_sustained'] = cart['gc_z'].rolling(6).mean()

mem_ts = cart.loc[cart['mem_sustained'] > 2.5, 'timestamp'].min()
gc_ts = cart.loc[cart['gc_sustained'] > 2.0, 'timestamp'].min()
print({'memory_sustained': mem_ts, 'gc_sustained': gc_ts})

In [ ]:
features = ['memory_usage_bytes', 'cpu_usage_percent', 'http_requests_per_sec', 'http_p99_latency_ms', 'http_5xx_rate', 'jvm_gc_pause_ms_avg']
X = StandardScaler().fit_transform(cart[features])
model = IsolationForest(contamination=0.05, random_state=42)
model.fit(X[:1440])
cart['if_score'] = -model.decision_function(X)
cart['if_label'] = model.predict(X)
print('first_if_anomaly', cart.loc[cart['if_label'] == -1, 'timestamp'].min())
cart.nlargest(10, 'if_score')[['timestamp', 'if_score']]


In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
ax[0].plot(cart['timestamp'], cart['memory_usage_bytes'], label='memory_usage_bytes')
ax[0].axhline(cart['memory_usage_bytes'].iloc[:720].mean() + 2 * cart['memory_usage_bytes'].iloc[:720].std(), color='red', linestyle='--', label='threshold')
ax[0].set_title('cart-service memory')
ax[1].plot(cart['timestamp'], cart['jvm_gc_pause_ms_avg'], label='jvm_gc_pause_ms_avg', color='orange')
ax[1].set_title('cart-service GC pause')
ax[2].plot(cart['timestamp'], cart['http_p99_latency_ms'], label='http_p99_latency_ms', color='green')
ax[2].set_title('cart-service p99 latency')
for a in ax:
    a.legend(loc='upper left')
    a.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(ART / 'metrics_anomaly_overview.png', dpi=160, bbox_inches='tight')
plt.show()


Kết luận chính:
- Memory anomaly bền vững bắt đầu trước khi alert nổ nhiều giờ.
- GC tăng sau memory.
- Latency và 5xx là hiệu ứng lan truyền sau đó.